# FLEURS dataset exploration

Explores the Google FLEURS dataset directly (full schema, not just the `audio`+`transcription`
fields the production `prep/` pipeline reads), downloads `CLIPS_PER_LANG` clips per language,
and writes a combined metadata table (origin, transcription, and whatever else FLEURS exposes).

In [ ]:
import json
import sys
import pathlib

import pandas as pd
import soundfile as sf

# Make the existing, tested `prep` package importable (it's only wired up via
# pytest.ini's pythonpath today, so add it to sys.path manually here).
sys.path.insert(0, str(pathlib.Path("../prep/src").resolve()))
from prep.slice import _to_samples  # reuse the tested audio-decoding helper

CLIPS_PER_LANG = 15
OUT_DIR = pathlib.Path("out")
CLIPS_DIR = OUT_DIR / "clips"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

## Language list

Reuse `prep/languages.json` as the single source of truth instead of duplicating it here.

In [ ]:
languages = json.load(open("../prep/languages.json"))["languages"]
pd.DataFrame(languages)

## Raw FLEURS schema introspection

The production pipeline only ever reads `audio` + `transcription`. Confirm live (not from
memory/docs) the full set of fields a FLEURS example actually carries, e.g. `id`, `num_samples`,
`path`, `raw_transcription`, `gender`, `lang_id`, `language`, `lang_group_id`.

In [ ]:
from datasets import Audio, load_dataset

probe_lang_id = languages[0]["lang_id"]
probe_ds = load_dataset("google/fleurs", probe_lang_id, split="train", streaming=True)
print("Features:", probe_ds.features)

# Cast audio to decode=False before iterating -- otherwise `datasets` tries to auto-decode
# with its new default backend, which requires the heavy optional `torchcodec` dependency
# (same issue prep/slice.py's _fleurs_stream already works around).
probe_ds = probe_ds.cast_column("audio", Audio(decode=False))
probe_example = next(iter(probe_ds))
print("\nFields present on a raw example:", sorted(probe_example.keys()))
for k, v in probe_example.items():
    if k == "audio":
        print(f"  {k}: <audio dict, keys={list(v.keys())}>")
    else:
        print(f"  {k}: {v!r}")

## Pull clips + full per-clip metadata

Streams each language directly (rather than going through `prep.slice.slice_clips`, which
only extracts `transcription`+audio and would drop the other fields surfaced above). Reuses
`prep.slice._to_samples` for the already-tested decode=False bytes-vs-array handling.

This FLEURS parquet has no page index and one giant row group per language (see
`prep/README.md`'s "Known risk" note), so reading *any* subset of a language's examples
already requires streaming close to the entire per-language file. Since a full pass is
unavoidable, we do single-pass **reservoir sampling** to pick `CLIPS_PER_LANG` uniformly
random examples (seeded, so it's reproducible) instead of just the first N -- same cost,
better sample.

Runs up to `MAX_CONCURRENT` languages at once. Each language still gets its own dedicated
thread with a hard wall-clock timeout (`LANGUAGE_TIMEOUT_SEC`): on a flaky/rate-limited
connection a single huge file can otherwise retry for hours (observed: 5+ hours stuck on
one language) before the library's own retry logic gives up on its own schedule. A stuck
thread is simply abandoned (it keeps running in the background and will eventually finish
or error on its own, but we stop waiting on it) so it can't hold up the other slots -- true
concurrency can briefly exceed `MAX_CONCURRENT` while stuck stragglers wind down, which is
an acceptable tradeoff for guaranteed forward progress. An `hf auth login` token (if set)
is picked up automatically by `datasets`/`huggingface_hub` for higher, less-throttled rate
limits -- no code change needed for that part.

In [ ]:
import random
import threading

from datasets import Audio

MAX_CONCURRENT = 4
LANGUAGE_TIMEOUT_SEC = 300  # 5 min hard cap per language; a stuck one gets skipped, not retried forever
SAMPLE_SEED = 42


def pull_language(lang, result):
    """Single-pass reservoir sampling of CLIPS_PER_LANG random examples for one language.
    Writes clip files to disk and stashes `("ok", rows)` or `("error", exc)` into `result`."""
    try:
        lang_id = lang["lang_id"]
        rng = random.Random(SAMPLE_SEED)

        ds = load_dataset("google/fleurs", lang_id, split="train", streaming=True)
        ds = ds.cast_column("audio", Audio(decode=False))

        reservoir = []  # raw `example` dicts (audio still encoded bytes, cheap to hold ~CLIPS_PER_LANG of)
        for idx, example in enumerate(ds):
            if idx < CLIPS_PER_LANG:
                reservoir.append(example)
            else:
                j = rng.randint(0, idx)
                if j < CLIPS_PER_LANG:
                    reservoir[j] = example

        rows = []
        for i, example in enumerate(reservoir):
            samples, sample_rate = _to_samples(example["audio"])
            out_path = CLIPS_DIR / f"{lang_id}_{i:02d}.wav"
            sf.write(out_path, samples, sample_rate)

            rows.append({
                "lang_id": lang_id,
                "language": lang["language"],
                "country": lang["country"],
                "continent": lang["continent"],
                "difficulty": lang["difficulty"],
                "id": example.get("id"),
                "num_samples": example.get("num_samples"),
                "gender": example.get("gender"),
                "transcription": example.get("transcription"),
                "raw_transcription": example.get("raw_transcription"),
                "sample_rate": sample_rate,
                "file_path": str(out_path),
            })
        result["status"], result["rows"] = "ok", rows
    except Exception as e:
        result["status"], result["error"] = "error", e


rows = []
failed_langs = []
completed = 0
state_lock = threading.Lock()          # guards rows/failed_langs/completed/CSV writes
admission = threading.Semaphore(MAX_CONCURRENT)  # bounds how many languages are in flight at once


def run_one(lang_num, lang):
    """Dispatched in its own thread; bounds a single language to LANGUAGE_TIMEOUT_SEC,
    then always releases its concurrency slot (even if the inner pull is abandoned still
    running) so a stuck language can't hold up the other MAX_CONCURRENT-1 slots."""
    global completed
    lang_id = lang["lang_id"]
    result = {}
    inner = threading.Thread(target=pull_language, args=(lang, result), daemon=True)
    inner.start()
    inner.join(timeout=LANGUAGE_TIMEOUT_SEC)

    with state_lock:
        if inner.is_alive():
            failed_langs.append(lang_id)
            print(f"[{lang_num}/{len(languages)}] {lang_id}: TIMED OUT after {LANGUAGE_TIMEOUT_SEC}s -- skipping")
        elif result.get("status") == "ok":
            rows.extend(result["rows"])
            print(f"[{lang_num}/{len(languages)}] {lang_id}: pulled {len(result['rows'])} clips")
        else:
            failed_langs.append(lang_id)
            print(f"[{lang_num}/{len(languages)}] {lang_id}: FAILED ({result.get('error')!r}) -- skipping")

        completed += 1
        # Checkpoint after every language so a crash partway through still leaves usable
        # partial output on disk instead of losing the whole run.
        pd.DataFrame(rows).to_csv(OUT_DIR / "fleurs_metadata.csv", index=False)
        print(f"  progress: {completed}/{len(languages)} languages processed")

    admission.release()


dispatchers = []
for lang_num, lang in enumerate(languages, start=1):
    admission.acquire()  # blocks here until a slot frees up (bounds true concurrency to MAX_CONCURRENT)
    th = threading.Thread(target=run_one, args=(lang_num, lang), daemon=True)
    th.start()
    dispatchers.append(th)

for th in dispatchers:
    th.join()

if failed_langs:
    print(f"\n{len(failed_langs)} language(s) failed/timed out: {failed_langs}")
else:
    print("\nAll languages pulled successfully.")

## Export + sanity check

In [ ]:
df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "fleurs_metadata.csv", index=False)

print(f"{len(df)} clips total")
display(df.groupby("language").size())
df.head(20)

## Optional extras

In [ ]:
import IPython.display as ipd

display(df["sample_rate"].value_counts())
display(df["transcription"].str.len().describe())

for lang_id, group in df.groupby("lang_id"):
    row = group.iloc[0]
    print(row["language"], "-", row["transcription"])
    samples, sample_rate = sf.read(row["file_path"])
    display(ipd.Audio(samples, rate=sample_rate))